---

# 03 Zero-Shot Text Analysis

---

This notebook demonstrates a zero-shot approach for analysing open-ended survey comments without requiring task-specific model training.

The objective is to enrich textual responses with interpretable features that can later be visualised in interactive dashboards and used for exploratory analytics.

The analysis focuses on:

- topic discovery,
- emotional dimensions,
- emotional granularity,
- contextual themes,
- and dashboard-ready exports.

The workflow combines modern language-model embeddings with unsupervised and zero-shot techniques to identify patterns in survey comments.

The resulting dataset can be used for:

- exploratory text analytics,
- dashboard development,
- emotion-aware visualisations,
- and downstream machine-learning experiments.

The notebook is designed as a reusable and privacy-preserving template that can be applied to any collection of open-ended survey responses.

In [1]:
# ============================================================
# Imports
# ============================================================

import pandas as pd
import numpy as np

from pathlib import Path
from tqdm import tqdm

from transformers import pipeline

In [2]:
# ============================================================
# Environment Check
# ============================================================

print("pandas:", pd.__version__)
print("numpy:", np.__version__)

model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

print("Environment successfully loaded.")
print("Embedding model:", model_name)

pandas: 2.3.3
numpy: 2.3.5
Environment successfully loaded.
Embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


In [3]:
# ============================================================
# Load Data
# ============================================================

DATA_PATH = Path(
    "../data/processed/synthetic_eda_ready.csv"
)

data = pd.read_csv(DATA_PATH)

print(data.shape)

data.head()

(7000, 79)


,start,ende,organisation,bereich,team_effizienz,staerken_nutzen,zusammenarbeit_team,psychologische_sicherheit,info_fuehrung_veraenderung,produktgestaltung_aktiv,...,fuehrung_score,strategie_score,arbeitsgestaltung_score,entwicklung_score,wertschoepfung_score,engagement_score,flag_speeder,answer_std,flag_straightliner,flag_quality
0,2024-10-10 12:37:00,2024-10-10 12:46:00,Zukunfts-Hub 02,FO-GA (Fokusbereich Gamma),4.0,2.0,2.0,3.0,3.0,2.0,...,3.000000,3.166667,3.000,3.285714,2.555556,2.8,False,0.980773,False,False
1,2024-05-20 19:11:00,2024-05-20 19:27:00,Inkubator-Einheit 04,KE-EP (Kernbereich Epsilon),2.0,2.0,1.0,1.0,1.0,1.0,...,1.166667,1.500000,1.250,1.285714,1.444444,1.4,False,0.745122,False,False
2,2024-04-10 13:14:00,2024-04-10 13:30:00,Spezialisten-Kreis 06,ZU-BE (Zukunftssegment Beta),5.0,3.0,5.0,5.0,5.0,4.0,...,4.333333,4.166667,3.625,3.571429,3.888889,4.0,False,0.883105,False,False
3,2025-01-15 14:57:00,2025-01-15 15:08:00,Fokus-Zelle 03,ZU-BE (Zukunftssegment Beta),4.0,5.0,3.0,5.0,5.0,5.0,...,4.666667,4.666667,4.625,4.571429,4.666667,4.6,False,0.719354,False,False
4,2024-03-25 20:19:00,2024-03-25 20:34:00,Fokus-Zelle 03,KE-EP (Kernbereich Epsilon),4.0,4.0,3.0,5.0,3.0,4.0,...,3.333333,3.666667,3.375,4.000000,4.000000,4.4,False,0.953129,False,False


In [4]:
# ============================================================
# Prepare Comments
# ============================================================

comments = (
    data["freitext_kommentar"]
    .fillna("")
    .astype(str)
    .str.strip()
)

data = data.loc[
    comments != ""
].copy()

print(
    f"Comments available: {len(data):,}"
)

Comments available: 5,000


***

# Textklassifizierung mit Zero-Shot & NLI

In diesem Notebook wird ein extrem vielseitiges KI-Modell als Baseline genutzt. Das Besondere daran: Texte lassen sich in beliebige Kategorien einteilen, ohne das Modell vorher mit spezifischen Beispielen trainieren zu müssen.

---

### Vorteile des Modells

* **Multilingualität:** Es unterstützt bis zu 100 Sprachen. Damit lassen sich deutsche, englische oder gemischte Texte problemlos und ohne zusätzlichen Übersetzungsaufwand analysieren.
* **Starke Basis:** Es basiert auf 'mDeBERTa-v3-base', einem der leistungsstärksten Sprachmodelle seiner Größenklasse.
* **Sofort einsatzbereit dank NLI-Training:** Durch ein spezielles Vortraining kann das Modell logische Schlüsse ziehen. Es versteht frei definierte Kategorien sofort, selbst wenn es diese zuvor noch nie gesehen hat.

---

### Das Funktionsprinzip (Natural Language Inference / NLI)

Das Modell nutzt die 'Natural Language Inference' (NLI) – also das logische Schließen. Dabei vergleicht es zwei Elemente miteinander:

1. **Die Prämisse (der Ausgangstext):** 
   > 'Ich habe gestern das Fußballspiel geschaut und es war unglaublich spannend!'
2. **Die Hypothese (die Kategorie, als Satz formuliert):** 
   > 'In diesem Text geht es um Sport.'

Das Modell entscheidet nun rein logisch, ob die Hypothese zum Text **passt** (trifft zu), **ihm widerspricht** (trifft nicht zu) oder **neutral** dazu steht. 

---

### Vorteile für die Praxis

Klassischerweise muss ein KI-Modell für jede neue Fragestellung mühsam mit vielen Beispielen trainiert werden (z. B. 'Ist dieser Text positiv oder negativ?' oder 'Geht es um Sport oder Politik?'). 

Durch das NLI-Prinzip entfällt dieser Schritt:
* Der Text wird einfach übergeben.
* Die gewünschten Kategorien lassen sich völlig frei definieren.
* Das Modell prüft logisch, welche Kategorie am besten passt – sofort, flexibel und ohne vorheriges Training auf den spezifischen Zieldaten.

In [5]:
# ============================================================
# Load Zero-Shot Model
# ============================================================

classifier = pipeline(
    "zero-shot-classification",
    model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli"
)

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [6]:
# ============================================================
# Analysis Labels
# ============================================================

# Barrett-inspired Core Affect dimensions.
# Phase 1 uses simple categorical labels that can later be mapped
# to dashboard coordinates.

# Valence:
# negative  <->  positive

valence_labels = [
    "positive",
    "neutral",
    "negative"
]

# Arousal:
# passive/deactivated  <->  highly activated

arousal_labels = [
    "high activation",
    "medium activation",
    "low activation"
]

# Coordinate mappings for later dashboard visualisations.

VALENCE_MAP = {
    "negative": -1.0,
    "neutral": 0.0,
    "positive": 1.0
}

AROUSAL_MAP = {
    "low activation": 0.0,
    "medium activation": 0.5,
    "high activation": 1.0
}

print("Valence labels:")
print(valence_labels)

print("\nArousal labels:")
print(arousal_labels)

Valence labels:
['positive', 'neutral', 'negative']

Arousal labels:
['high activation', 'medium activation', 'low activation']


In [7]:
# ============================================================
# Zero-Shot Classification
# ============================================================

# Analyse each comment along Barrett's two Core Affect dimensions:
# - Valence
# - Arousal

results = []

for comment in tqdm(
    data["freitext_kommentar"],
    desc="Running Zero-Shot Analysis"
):

    # --------------------------------------------------------
    # Valence
    # --------------------------------------------------------

    valence_prediction = classifier(
        comment,
        valence_labels
    )

    valence_label = valence_prediction["labels"][0]
    valence_confidence = valence_prediction["scores"][0]

    # --------------------------------------------------------
    # Arousal
    # --------------------------------------------------------

    arousal_prediction = classifier(
        comment,
        arousal_labels
    )

    arousal_label = arousal_prediction["labels"][0]
    arousal_confidence = arousal_prediction["scores"][0]

    # --------------------------------------------------------
    # Store result
    # --------------------------------------------------------

    results.append(
        {
            "valence_label": valence_label,
            "valence_confidence": valence_confidence,
            "valence_score": VALENCE_MAP[valence_label],

            "arousal_label": arousal_label,
            "arousal_confidence": arousal_confidence,
            "arousal_score": AROUSAL_MAP[arousal_label],
        }
    )

results = pd.DataFrame(results)

print(
    f"Analysed {len(results):,} comments."
)

Running Zero-Shot Analysis: 100%|██████████| 5000/5000 [32:30<00:00,  2.56it/s] 

Analysed 5,000 comments.


In [8]:
# ============================================================
# Merge Results
# ============================================================

data = data.reset_index(drop=True)

data["valence_label"] = results["valence_label"]
data["valence_confidence"] = results["valence_confidence"]
data["valence_score"] = results["valence_score"]

data["arousal_label"] = results["arousal_label"]
data["arousal_confidence"] = results["arousal_confidence"]
data["arousal_score"] = results["arousal_score"]

display(
    data[
        [
            "freitext_kommentar",
            "valence_label",
            "valence_score",
            "arousal_label",
            "arousal_score"
        ]
    ].head()
)

,freitext_kommentar,valence_label,valence_score,arousal_label,arousal_score
0,Im Kontext 'Wissensaustausch und Lernen' werde...,negative,-1.0,medium activation,0.5
1,Im Kontext 'Kollaborativer Arbeitsmodus' werde...,positive,1.0,medium activation,0.5
2,Im Kontext 'Wissensaustausch und Lernen' werde...,positive,1.0,medium activation,0.5
3,Im Kontext 'Reflektion und Verbesserung' werde...,positive,1.0,medium activation,0.5
4,Im Kontext 'Kollaborativer Arbeitsmodus' werde...,neutral,0.0,medium activation,0.5


In [9]:
# ============================================================
# Review Results
# ============================================================

print("Valence distribution:")

display(
    data["valence_label"]
    .value_counts()
    .to_frame("count")
)

print("\nArousal distribution:")

display(
    data["arousal_label"]
    .value_counts()
    .to_frame("count")
)

Valence distribution:


,count
valence_label,
positive,3114
negative,1146
neutral,740



Arousal distribution:


,count
arousal_label,
medium activation,3599
high activation,873
low activation,528


In [10]:
pd.Series(data.columns)

0                  start
1                   ende
2           organisation
3                bereich
4         team_effizienz
             ...        
80    valence_confidence
81         valence_score
82         arousal_label
83    arousal_confidence
84         arousal_score
Length: 85, dtype: object

In [11]:
# ============================================================
# Quality Metrics
# ============================================================

metrics = {}

# ------------------------------------------------------------
# Coverage
# ------------------------------------------------------------

metrics["total_comments"] = len(data)

metrics["classified_comments"] = (
    data["valence_label"]
    .notna()
    .sum()
)

metrics["coverage_pct"] = round(
    metrics["classified_comments"]
    / metrics["total_comments"]
    * 100,
    2
)

# ------------------------------------------------------------
# Confidence
# ------------------------------------------------------------

metrics["avg_valence_confidence"] = round(
    data["valence_confidence"].mean(),
    3
)

metrics["avg_arousal_confidence"] = round(
    data["arousal_confidence"].mean(),
    3
)

# ------------------------------------------------------------
# Distribution Balance
# ------------------------------------------------------------

metrics["n_positive"] = (
    data["valence_label"]
    .eq("positive")
    .sum()
)

metrics["n_neutral"] = (
    data["valence_label"]
    .eq("neutral")
    .sum()
)

metrics["n_negative"] = (
    data["valence_label"]
    .eq("negative")
    .sum()
)

metrics_df = pd.DataFrame(
    metrics.items(),
    columns=["metric", "value"]
)

display(metrics_df)

,metric,value
0,total_comments,5000.000
1,classified_comments,5000.000
2,coverage_pct,100.000
3,avg_valence_confidence,0.738
4,avg_arousal_confidence,0.583
5,n_positive,3114.000
6,n_neutral,740.000
7,n_negative,1146.000


In [12]:
# ============================================================
# Export Results
# ============================================================

export_columns = [
    "iteration_id",
    "bereich_ebene",
    "freitext_kommentar",

    "valence_label",
    "valence_score",
    "valence_confidence",

    "arousal_label",
    "arousal_score",
    "arousal_confidence"
]

output = data[export_columns]

output.to_csv(
    "../data/results/zero_shot_core_affect.csv",
    index=False,
    encoding="utf-8-sig",
    decimal=','
)

print(
    "File exported: zero_shot_core_affect.csv"
)



File exported: zero_shot_core_affect.csv


# Erste Erkenntnisse aus der Zero-Shot Core-Affect-Analyse

## Technischer Durchstich erfolgreich

Die erste Zero-Shot-Pipeline konnte erfolgreich umgesetzt werden.

Folgende Schritte funktionieren bereits Ende-zu-Ende:

- Laden der bereinigten Kommentardaten
- Zero-Shot-Klassifikation von Valenz und Aktivierung
- Ableitung von Core-Affect-Koordinaten
- Speicherung der Ergebnisse für spätere Dashboard-Analysen

Damit ist die technische Machbarkeit des geplanten Analyseansatzes nachgewiesen.

---

## Erste Beobachtungen

- Die Valenz-Klassifikation liefert überwiegend plausible Ergebnisse.
- Positive Kommentare werden meist als positiv erkannt.
- Kritische Kommentare werden häufig als negativ erkannt.
- Aktivierung scheint schwieriger zu klassifizieren als Valenz.
- Einzelne stark emotionale Kommentare werden aktuell teilweise als „neutral“ oder „medium activation“ eingestuft.

---

## Bezug zu Barrett

Der aktuelle Ansatz bildet bereits die beiden Kern-Dimensionen des Core-Affect-Modells ab:

- **Valenz** (angenehm ↔ unangenehm)
- **Aktivierung** (niedrig ↔ hoch)

Damit lassen sich Kommentare bereits in einem Core-Affect-Grid visualisieren.

Die aktuelle Implementierung ist jedoch noch eine vereinfachte Annäherung und keine vollständige Operationalisierung der Theorie von Lisa Feldman Barrett.

---

## Aktuelle Grenzen

- Die Zuordnung erfolgt über diskrete Kategorien statt kontinuierlicher Skalen.
- Emotionale Granularität wird noch nicht berücksichtigt.
- Body-Budget-Indikatoren werden noch nicht analysiert.
- Themen und Auslöser emotionaler Zustände werden noch nicht automatisch erkannt.
- Die Güte der Aktivierungsdimension muss noch genauer überprüft werden.

---

## Fazit

Der erste Prototyp zeigt, dass offene Kommentare automatisch mit emotionalen Informationen angereichert werden können.

Die Ergebnisse eignen sich bereits als Grundlage für:

- ein erstes Core-Affect-Dashboard,
- die Visualisierung von Valenz und Aktivierung,
- sowie die weitere Entwicklung von Themenclustern, Body-Budget-Indikatoren und emotionaler Granularität.

